# Day 5: Silver Standardization (OMOP CDM v5.4 & Entity Linkage)

This notebook demonstrates probabilistic patient linkage using Splink and PySpark across disparate EHR records, and OMOP mapping.

In [ ]:
from pyspark.sql import SparkSession
import splink.spark.comparison_library as cl
from splink.spark.linker import SparkLinker
from pyspark.sql.functions import col, lit

spark = SparkSession.builder.appName("Day05_OMOP_Linkage").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark session created.")

In [ ]:
# Synthetic data from two different systems
data_sys1 = [
    ("A1", "John", "Doe", "1980-01-01", "M"),
    ("A2", "Jane", "Smith", "1990-05-15", "F")
]
data_sys2 = [
    ("B1", "Jon", "Doe", "1980-01-01", "M"), # Typo in first name
    ("B2", "Jane", "Smith", "1990-05-15", "F")
]

df1 = spark.createDataFrame(data_sys1, ["unique_id", "first_name", "last_name", "dob", "gender"])
df2 = spark.createDataFrame(data_sys2, ["unique_id", "first_name", "last_name", "dob", "gender"])

# Add source column for splink
df1 = df1.withColumn("source_dataset", lit("sys1"))
df2 = df2.withColumn("source_dataset", lit("sys2"))

combined_df = df1.union(df2)
combined_df.show()

In [ ]:
# Splink linkage model settings
settings = {
    "link_type": "link_and_dedupe",
    "blocking_rules_to_generate_predictions": [
        "l.dob = r.dob",
        "l.last_name = r.last_name"
    ],
    "comparisons": [
        cl.levenshtein_at_thresholds("first_name", 2),
        cl.exact_match("last_name"),
        cl.exact_match("dob"),
        cl.exact_match("gender")
    ],
    "retain_matching_columns": True,
    "retain_intermediate_calculation_columns": True
}

linker = SparkLinker(combined_df, settings, spark=spark)
linker.estimate_u_using_random_sampling(max_pairs=1e6)

# Assuming m-probabilities for demonstration without full EM training
linker.estimate_m_from_label_column("first_name") # dummy training

df_predict = linker.predict(threshold_match_probability=0.9)

print("Probabilistic Linkage Results:")
df_predict.as_pandas_dataframe()[['unique_id_l', 'unique_id_r', 'match_probability']]